[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/coating/coating.ipynb)

# Optical Coatings

This example computes reflection, transmission, internal waveforms, and time-domain pulse propagation across stratified optical coatings, cholesteric liquid-crystal helices, and axion interfaces. A quarter-wave dielectric film cancels surface reflection by destructive interference, a twisted uniaxial crystal reflects light of one circular handedness while passing the other, and a jump in magnetoelectric axion coupling rotates polarization at an interface where the refractive index does not change.

Each material is modelled as a constitutive map from spacetime field bivectors to excitation antibivectors, built from electric worldsheet dyads `axes & time` and spatial magnetic patches, and twisting a crystal optic axis transports that map by a spatial rotor, `rotors >> crystal(rotors << Field)`. Taking tangential traces at an interface packs field and excitation continuity into one four-component boundary state, solving the longitudinal Maxwell constraints alongside those traces reconstructs the full field bivector, and matching against the incident exterior over `Polarization` solves reflection and transmission for every polarization at once. Pairing the stress-energy current `(fields & excitations) / 2 * observer - ((fields & observer) ^ excitations)` with the interface normal measures optical power, and the same constitutive maps step real staggered fields through time ([Berreman (1972)](https://doi.org/10.1364/JOSA.62.000502)).

In Berreman's $4 \times 4$ transfer-matrix notation, electric and magnetic vectors satisfy split constitutive blocks $\mathbf{D} = \mathbf{R}(\phi)\boldsymbol{\varepsilon}\mathbf{R}(\phi)^{\mathsf{T}}\mathbf{E} + \boldsymbol{\xi}\mathbf{H}$ and $\mathbf{B} = \boldsymbol{\zeta}\mathbf{E} + \boldsymbol{\mu}\mathbf{H}$, and eliminating $E_z$ and $H_z$ from $\nabla \times \mathbf{E} = i\omega\mathbf{B}$ and $\nabla \times \mathbf{H} = -i\omega\mathbf{D}$ by Schur complement on a $6 \times 6$ coordinate matrix assembles $\partial_z \boldsymbol{\Psi} = i k_0 \boldsymbol{\Delta}\,\boldsymbol{\Psi}$ for the ordered state vector $\boldsymbol{\Psi} = (E_x, H_y, E_y, -H_x)^{\mathsf{T}}$. Matching incident and transmitted amplitudes solves a $4 \times 4$ linear system for $2 \times 2$ Jones matrices, while normal power is evaluated separately from the Poynting cross product $\tfrac{1}{2}\operatorname{Re}(\mathbf{E} \times \mathbf{H}^*) \cdot \hat{\mathbf{z}}$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from functools import reduce
from itertools import accumulate, islice

import numpy as np

from numga import Algebra, NumpyContext, Orientation, stack
from numga.algebras import STA as ga
from numga.sparse import Sparse
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.optics.coating import render

mv = NumpyContext(ga, dtype=np.complex128).multivector
# Real spacetime context for stepping real fields along z without harmonic phase factors.
real = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                       # Scalar+
Wavevector = ga.gatype.vector()                                                   # Vector+
Field = ga.gatype.bivector()                                                      # Bivector+
Excitation = ga.gatype.antibivector()                                             # Antibivector-
Direction = ga.gatype.antivector()                                                # Antivector-
Polarization = Field.derive.touching(Algebra("x+y+"), Algebra("t-"))              # Bivector+
Boundary = ga.gatype(ga.subspace("x y txz tyz"), orientation=Orientation.extrinsic)  # Odd+
Constitutive = ga.gatype((Excitation, Field))                                     # Excitation- <- Field+
Inverse = ga.gatype((Field, Excitation))                                          # Field+ <- Excitation-
FieldMap = ga.gatype((Field, Boundary))                                           # Field+ <- Boundary+
Propagation = ga.gatype((Boundary, Boundary))                                     # Boundary+ <- Boundary+
PolarizationMap = ga.gatype((Polarization, Polarization))                         # Polarization+ <- Polarization+
WaveMap = ga.gatype((Field, Polarization))                                        # Field+ <- Polarization+
Outgoing = ga.gatype((Boundary, Polarization))                                    # Boundary+ <- Polarization+

# Observer rest-frame 4-velocity, spatial principal axes, and spatial magnetic bivector planes.
time = -mv.t.dual()                                                               # Direction-
axes = stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))                             # [3] Direction-
magnetic_planes = stack((mv.yz, mv.zx, mv.xy))                                    # [3] Field+
swept_planes = axes & time                                                        # [3] Excitation-
patches = magnetic_planes.dual()                                                  # [3] Excitation-
unit_frequency = -mv.t                                                            # Wavevector+
field_boundary = mv.t ^ (mv.t.inverse() | (mv.z ^ Field))                         # Boundary+ <- Field+
excitation_boundary = mv.t.inverse() | (mv.t ^ (mv.z ^ Excitation).dual())        # Boundary+ <- Excitation-
electric_trace = mv.z.inverse() | Boundary                                        # Polarization+ <- Boundary+

## Constitutive response and boundary operators

A monochromatic field bivector combines electric worldsheet planes and spatial magnetic planes with complex phase coefficients whose positive imaginary phase grows in time. Each dielectric maps a field bivector to its excitation antibivector through principal electric worldsheet dyads and magnetic patch dyads in vacuum units. At an interface normal to `z`, four tangential combinations of field and excitation stay continuous while two derivative-free longitudinal Maxwell equations vanish identically, reconstructing the full six-component field bivector and its normal propagation generator from the four-component boundary state.

In [ ]:
substrate_index = 1.5
substrate_permittivity = np.full(3, substrate_index ** 2)
reluctivity = 1.0
parallel = unit_frequency

In [ ]:
def dielectric_response(permittivity: np.ndarray, reluctivity: np.ndarray) -> Constitutive:
    """Principal `permittivity` (`[..., 3]`) and scalar `reluctivity` (`[...]`) in vacuum units assemble the rest-frame constitutive dyad sum (`[...] Excitation- <- Field+`)."""
    # Time squares to minus one, so electric worldsheet dyads enter with a minus sign.
    electric = -(permittivity * swept_planes * (Field & swept_planes)).sum(axis=-1)
    magnetic = (patches * (Field & patches)).sum(axis=-1)
    return electric + reluctivity * magnetic                                      # [...] Excitation- <- Field+


def layer_operators(response: Constitutive, parallel: Wavevector) -> tuple[FieldMap, Propagation]:
    """Reducing Maxwell's equations in a layer with constitutive map `response` (`[...] Excitation- <- Field+`) normal to `z` at tangential wavevector `parallel` yields the full-field reconstruction map and the normal propagation generator."""
    # Four tangential combinations of field and excitation stay continuous across interfaces.
    boundary = field_boundary + excitation_boundary(response)                     # [...] Boundary+ <- Field+
    # Two longitudinal Maxwell equations contain no normal derivative and vanish identically.
    constraint = (-(mv.z | (mv.z ^ (parallel ^ Field)))
                  - mv.z * (mv.z | (parallel ^ response).dual()))
    reconstruction = (boundary + constraint).solve(Boundary)                      # [...] Field+ <- Boundary+
    # Contracting parallel ^ F + z ^ dF/dz == 0 with z.inverse() gives the normal derivative of the boundary state.
    generator = -(field_boundary(mv.z.inverse() | (parallel ^ reconstruction))
                  + excitation_boundary(mv.z.inverse() | (parallel ^ response(reconstruction))))
    return reconstruction, generator                                              # [...] Field+ <- Boundary+, Boundary+ <- Boundary+


substrate_response = dielectric_response(substrate_permittivity, reluctivity)     # [] Excitation- <- Field+

## Mode propagation and interface scattering

Across a homogeneous layer, each eigenmode of the propagation generator acquires its own phase factor. Because anisotropic eigenmodes need not be orthogonal, solving the summed mode-dyad frame supplies the biorthogonal reciprocal reading each mode's amplitude, carrying boundary states from a layer's exit face back to its entrance face so adjacent layers compose in physical order. In each isotropic exterior, forward and backward plane waves carry opposite normal propagation indices; matching the entrance boundary state over `Polarization` solves transmission and reflection for all incident polarizations at once, and pairing the stress-energy current across both oscillation quadratures with `z` gives the optical power.

In [ ]:
def layer_propagation(generator: Propagation, optical_distance: np.ndarray) -> Propagation:
    """Exponentiating `generator` (`[...] Boundary+ <- Boundary+`) over `optical_distance` in its eigenmodes builds the transfer map carrying boundary states from a layer's exit face back to its entrance face."""
    indices, modes = generator.eig()                                              # [..., 4] Scalar+, Boundary+
    frame = (modes * modes.scalar_product(Boundary)).sum(axis=-1)                 # [...] Boundary+ <- Boundary+
    # Solving the mode-dyad frame supplies the biorthogonal reciprocal reading each mode's amplitude.
    reciprocal = frame[..., None].solve(modes)                                    # [..., 4] Boundary+
    phases = (indices * (1j * optical_distance[..., None])).exp()                 # [..., 4] Scalar+
    return (phases * modes * reciprocal.scalar_product(Boundary)).sum(axis=-1)    # [...] Boundary+ <- Boundary+


def stack_propagation(layers: Iterable[Propagation]) -> Propagation:
    """Composing `layers` ordered from incident medium to substrate builds the transfer map of the full stack."""
    return reduce(lambda total, layer: total(layer), layers)                      # [...] Boundary+ <- Boundary+

In [ ]:
def exterior_wave(index: float, parallel: Wavevector) -> WaveMap:
    """Completing the 4-potential perpendicular to the ray builds the forward exterior plane-wave map (`[...] Field+ <- Polarization+`) at refractive `index` and tangential wavevector `parallel`."""
    normal_index = (index ** 2 - (parallel - unit_frequency).squared()).square_root()
    # Undo the temporal wedge and add a normal part so wavevector ^ potential has Polarization as its tangential electric trace.
    tangential = unit_frequency.inverse() | Polarization                          # Wavevector+ <- Polarization+
    potential = tangential - mv.z * (parallel | tangential) / normal_index
    wavevector = parallel + mv.z * normal_index
    return wavevector ^ potential                                                 # [...] Field+ <- Polarization+


def interface_scattering(
    entrance: Outgoing,
    incident_response: Constitutive,
    incident_index: float,
    parallel: Wavevector,
) -> tuple[PolarizationMap, PolarizationMap]:
    """Matching an `entrance` boundary map (`[...] Boundary+ <- Polarization+`) against an exterior medium of `incident_response` and `incident_index` at `parallel` solves for the polarization reflection and transmission maps."""
    _, generator = layer_operators(incident_response, parallel)
    normal_index = (incident_index ** 2 - (parallel - unit_frequency).squared()).square_root()
    incoming = electric_trace((Boundary + generator / normal_index) / 2)          # [...] Polarization+ <- Boundary+
    returning = electric_trace((Boundary - generator / normal_index) / 2)         # [...] Polarization+ <- Boundary+
    transmission = incoming(entrance).inverse()                                   # [...] Polarization+ <- Polarization+
    reflection = returning(entrance)(transmission)                                # [...] Polarization+ <- Polarization+
    return reflection, transmission

In [ ]:
def wave_power(field: Field, response: Constitutive) -> Scalar:
    """Pairing the stress-energy current of `field` (`[...] Field+`) in `response` (`[...] Excitation- <- Field+`) with the interface normal `z` sums the optical power (`[...] Scalar+`) over both real quadratures of an oscillation cycle."""
    excitation = response(field)                                                  # [...] Excitation-
    fields = stack([field.real(), (-1j * field).real()])                          # [2, ...] Field+
    excitations = stack([excitation.real(), (-1j * excitation).real()])           # [2, ...] Excitation-
    observer, face = time.real(), mv.z.real()                                     # Direction-, Wavevector+
    currents = (fields & excitations) / 2 * observer - ((fields & observer) ^ excitations)  # [2, ...] Direction-
    return (face & currents).sum(axis=0)                                          # [...] Scalar+


def scattering_powers(
    reflection: PolarizationMap,
    transmission: PolarizationMap,
    polarization: Polarization,
    incident_wave: WaveMap,
    incident_response: Constitutive,
    substrate_wave: WaveMap,
    substrate_response: Constitutive,
) -> tuple[Scalar, Scalar]:
    """Normalizing reflected and transmitted wave powers by the incident wave power returns the reflectance and transmittance (`[...] Scalar+`, `[...] Scalar+`) for `polarization`."""
    incident_power = wave_power(incident_wave(polarization), incident_response)
    reflectance = wave_power(incident_wave(reflection(polarization)), incident_response) / incident_power
    transmittance = wave_power(substrate_wave(transmission(polarization)), substrate_response) / incident_power
    return reflectance, transmittance

## Quarter-wave antireflection films

At normal incidence, a quarter-wave dielectric film cancels surface reflection at its design wavelength when its refractive index equals the geometric mean of the incident and substrate indices. Setting film thickness to zero recovers bare glass from the same layer transfer map.

In [ ]:
design_wavelength = 550.0
n_wavelengths = 401
wavelengths = np.linspace(450, 850, n_wavelengths)
incident_index = 1.0
coating_indices = np.array([substrate_index, np.sqrt(incident_index * substrate_index), 1.38])
optical_thicknesses = np.array([0.0, design_wavelength / 4, design_wavelength / 4])
coating_thicknesses = optical_thicknesses / coating_indices

In [ ]:
incident_response = dielectric_response(np.full(3, incident_index ** 2), reluctivity)
incident_wave = exterior_wave(incident_index, parallel)
substrate_wave = exterior_wave(substrate_index, parallel)
substrate_outgoing = (field_boundary + excitation_boundary(substrate_response))(substrate_wave)
film_response = dielectric_response(coating_indices[:, None] ** 2 * np.ones(3), reluctivity)
_, film_generator = layer_operators(film_response, parallel)
distance = 2 * np.pi * coating_thicknesses[:, None] / wavelengths
coating = layer_propagation(film_generator[:, None], distance)                    # [3, n_wavelengths] Boundary+ <- Boundary+
reflection, transmission = interface_scattering(coating(substrate_outgoing), incident_response, incident_index, parallel)
matching_reflection, matching_transmission = scattering_powers(
    reflection, transmission, mv.xt, incident_wave, incident_response, substrate_wave, substrate_response
)
save_figure(render.spectra_figure(wavelengths, matching_reflection), "coating_chi_matching");

Bare glass in blue reflects four percent across the visible range, the ideal quarter-wave film in orange drops to zero at the 550 nm design wavelength, and magnesium fluoride in teal reduces reflectance to just over one percent.

## Handedness-selective Bragg reflection

A uniaxial crystal responds differently along its optic axis and perpendicular to it. Twisting the optic axis from slice to slice transports the constitutive map by a spatial rotor, `rotors >> crystal(rotors << Field)`, coupling the two transverse polarizations to open a Bragg reflection band for the circular input turning in the same sense as the helix ([cholesteric optical reflectors](https://www.nature.com/articles/nmat2045)).

In [ ]:
ordinary_index = 1.5
extraordinary_index = 1.8
surrounding_index = (ordinary_index + extraordinary_index) / 2
pitch = 400.0
n_turns = 6
n_slices_per_turn = 16
n_layers = n_turns * n_slices_per_turn
slice_thickness = pitch / n_slices_per_turn
slice_depths = (np.arange(n_layers) + 0.5) * slice_thickness
crystal_permittivity = np.array([extraordinary_index ** 2, ordinary_index ** 2, ordinary_index ** 2])
twist_signs = np.array([1.0, -1.0])
polarization_signs = np.array([1.0, -1.0])

In [ ]:
def helix_media(depths: np.ndarray) -> tuple[Direction, Constitutive]:
    """Transporting a uniaxial dielectric across `depths` (`[n_depths]`) by opposite spatial rotors yields the crystal optic axes (`[n_depths, 2] Direction-`) and constitutive maps (`[n_depths, 2] Excitation- <- Field+`)."""
    angles = 2 * np.pi * depths[:, None] / pitch * twist_signs
    rotors = (mv.xy * (-angles / 2)).exp()                                        # [..., 2] Rotor+
    crystal = dielectric_response(crystal_permittivity, reluctivity)              # [] Excitation- <- Field+
    return rotors >> mv.x.dual(), rotors >> crystal(rotors << Field)


_, helix_response = helix_media(slice_depths)
_, helix_generator = layer_operators(helix_response, parallel)
distance = 2 * np.pi * slice_thickness / wavelengths
layers = layer_propagation(helix_generator[..., None], distance)                  # [n_layers, 2, n_wavelengths] Boundary+ <- Boundary+
propagation = stack_propagation(layers)                                           # [2, n_wavelengths] Boundary+ <- Boundary+
surrounding_response = dielectric_response(np.full(3, surrounding_index ** 2), reluctivity)
surrounding_wave = exterior_wave(surrounding_index, parallel)
surrounding_outgoing = (field_boundary + excitation_boundary(surrounding_response))(surrounding_wave)
reflection, transmission = interface_scattering(propagation(surrounding_outgoing), surrounding_response, surrounding_index, parallel)
circular_polarizations = (mv.xt + mv.yt * (1j * polarization_signs)) / np.sqrt(2)  # [2] Polarization+
twisted_reflection, twisted_transmission = scattering_powers(
    reflection, transmission, circular_polarizations[:, None, None],
    surrounding_wave, surrounding_response, surrounding_wave, surrounding_response,
)                                                                                 # [2, 2, n_wavelengths] Scalar+, Scalar+
save_figure(render.handedness_figure(wavelengths, twisted_reflection), "coating_handedness");

In the left panel for the positive crystal twist, the co-rotating circular input turning from x toward y in blue peaks near complete reflection across the Bragg band around 660 nm while the counter-rotating input in orange passes through; reversing the crystal twist in the right panel exchanges the blue and orange spectra.

## Oblique illumination

Tilting the incident beam adds a spatial component along x to the tangential wavevector and rotates both circular electric planes with the ray before taking their tangential traces.

In [ ]:
n_angles = 41
incidence_angles = np.linspace(0, np.deg2rad(70), n_angles)
n_map_wavelengths = 121
map_wavelengths = np.linspace(450, 850, n_map_wavelengths)

In [ ]:
parallel_oblique = unit_frequency + mv.x * (surrounding_index * np.sin(incidence_angles))
_, oblique_generator = layer_operators(helix_response[:, 0, None], parallel_oblique)
distance = 2 * np.pi * slice_thickness / map_wavelengths
oblique_propagation = stack_propagation(
    layer_propagation(generator[:, None], distance) for generator in oblique_generator
)                                                                                 # [n_angles, n_map_wavelengths] Boundary+ <- Boundary+
oblique_wave = exterior_wave(surrounding_index, parallel_oblique[:, None])
oblique_outgoing = (field_boundary + excitation_boundary(surrounding_response))(oblique_wave)
reflection, transmission = interface_scattering(
    oblique_propagation(oblique_outgoing), surrounding_response, surrounding_index, parallel_oblique[:, None]
)
tilt = (mv.zx * (incidence_angles / 2)).exp()                                     # [n_angles] Rotor+
oblique_fields = tilt[None, :] >> circular_polarizations[:, None]                 # [2, n_angles] Field+
tangential_fields = mv.z.inverse() | (mv.z ^ oblique_fields)                      # [2, n_angles] Polarization+
angle_reflection, angle_transmission = scattering_powers(
    reflection, transmission, tangential_fields[..., None],
    oblique_wave, surrounding_response, oblique_wave, surrounding_response,
)
save_figure(render.angular_figure(map_wavelengths, incidence_angles, angle_reflection), "coating_polarization_map");

The left panel maps reflectance for the input turning from x toward y and the right panel maps the opposite circular input across the positive helix, from zero reflectance in black to unit reflectance in bright yellow-white. Increasing the incidence angle shifts the Bragg stopband toward shorter wavelengths while grazing reflection grows in both channels.

## Internal waveforms

Propagating the transmitted exit boundary state backward through the stack reconstructs the full six-component field bivector at interior sample depths across every slice.

In [ ]:
band_wavelength = pitch * surrounding_index
n_samples_per_layer = 5
fractions = np.linspace(0, 1, n_samples_per_layer)

In [ ]:
def interior_fields(
    reconstruction: FieldMap,
    generator: Propagation,
    thicknesses: np.ndarray,
    wavelength: float,
    exit_state: Boundary,
    fractions: np.ndarray,
) -> Field:
    """Propagating `exit_state` (`[...] Boundary+`) backward across layers of `thicknesses` (`[n_layers]`) at `wavelength` reconstructs the full field bivectors (`[..., n_layers, n_samples] Field+`) at fractional layer depths `fractions` (`[n_samples]`)."""
    optical_distances = 2 * np.pi * thicknesses / wavelength
    crossings = layer_propagation(generator, optical_distances)                   # [...] Boundary+ <- Boundary+
    # Accumulate layer crossings from the substrate back so each layer knows the boundary state at its exit face.
    faces = accumulate(crossings[:0:-1], lambda state, crossing: crossing(state), initial=exit_state)
    right_faces = stack(tuple(faces)[::-1], axis=-1)                              # [...] Boundary+
    inward = layer_propagation(generator[:, None], optical_distances[:, None] * (1 - fractions))  # [...] Boundary+ <- Boundary+
    return reconstruction[:, None](inward(right_faces[..., None]))                # [...] Field+


field_depths = (np.arange(n_layers)[:, None] + fractions) * slice_thickness
field_reconstruction, field_generator = layer_operators(helix_response[:, 0], parallel)
field_layers = layer_propagation(field_generator, np.full(n_layers, 2 * np.pi * slice_thickness / band_wavelength))
_, field_transmission = interface_scattering(
    stack_propagation(field_layers)(surrounding_outgoing), surrounding_response, surrounding_index, parallel
)
exit_state = surrounding_outgoing(field_transmission(circular_polarizations))     # [2] Boundary+
interior = interior_fields(
    field_reconstruction, field_generator, np.full(n_layers, slice_thickness), band_wavelength, exit_state, fractions
)                                                                                 # [2, n_layers, n_samples_per_layer] Field+
save_figure(render.internal_field_figure(field_depths, interior), "coating_internal_field");

Red and blue mark opposite signs of the transverse electric component along x over two optical cycles. At the Bragg band centre, the co-rotating wave in the left panel forms stationary crests whose amplitude decays across the lossless helix as power reflects back toward the entrance, whereas the counter-rotating wave in the right panel travels through at constant amplitude.

## Time-domain pulse propagation

Stepping real field bivectors directly through time requires no complex phase factors. Electric planes sit on the vertices of a closed loop and magnetic planes sit halfway between neighbours on the edges, turning the exterior Maxwell equations `d ^ field == 0` and `d ^ excitation == 0` into a staggered leapfrog. Contracting with the inverse time 1-form turns each temporal derivative into a difference along `z`, where the edge coboundary and its adjugate form the energy-conserving pair. Ray-tracing the 3D pulse animation below takes roughly ten seconds.

In [ ]:
node_spacing = 40.0
n_nodes = 1000
slab_start = 18000.0
pulse_start = 10000.0
pulse_width = 1000.0
hand_signs = np.array([1.0, -1.0])
courant = 1.0
light_travel = 27000.0
interval = courant * node_spacing
n_steps = round(light_travel / interval)
n_frames = 168
frame_duration_ms = 40

In [ ]:
def leapfrog_states(
    difference: Sparse,
    node_inverse: Inverse,
    edge_response: Constitutive,
    electric: Field,
    magnetic: Field,
    spacing: float,
    interval: float,
    n_steps: int,
) -> Iterator[tuple[Field, Field]]:
    """Yields staggered `(electric, magnetic)` field states (`[...] Field+[V]`, `[...] Field+[E]`) across `n_steps` leapfrog updates of `d ^ F == 0` and `d ^ G == 0` on a 1D periodic loop."""
    rate = interval / spacing
    for _ in range(n_steps):
        # Faraday update d ^ F == 0 advances edge magnetic planes from neighbour electric differences.
        magnetic = magnetic - (real.t.inverse() | (real.z ^ (difference * electric))) * rate
        excitation = edge_response(magnetic)
        # Ampere-Maxwell update d ^ G == 0 pulls edge excitations back onto the nodes via the adjugate difference.
        displacement = (real.t.inverse() | (real.z ^ (difference.adjugate() * excitation))) * rate
        electric = electric + node_inverse(displacement)
        yield electric, magnetic


def electromagnetic_energy(
    electric: Field,
    magnetic: Field,
    node_response: Constitutive,
    edge_response: Constitutive,
    spacing: float,
) -> Scalar:
    """Pairing `electric` (`[...] Field+[V]`) on the nodes and `magnetic` (`[...] Field+[E]`) on the staggered edges with their excitations returns the electromagnetic energy (`[...] Scalar+[V]`) of each staggered interval of length `spacing`."""
    # Spatial magnetic bivectors pair with their excitations to minus twice the magnetic energy density.
    return ((electric & node_response(electric)) - (magnetic & edge_response(magnetic))) * (spacing / 2)

In [ ]:
def initial_pulse(positions: np.ndarray) -> Field:
    """Gaussian wavepackets at `positions` (`[V]`) build co-rotating, counter-rotating, and linearly polarized electric fields (`[3, V] Field+`)."""
    envelope = np.exp(-((positions - pulse_start) / pulse_width) ** 2 / 2)
    phases = 2 * np.pi * surrounding_index * positions / band_wavelength
    rotors = (real.xy * (-hand_signs[:, None] * phases / 2)).exp()                # [2, ...] Rotor+
    carriers = rotors >> real.xt                                                  # [2, ...] Field+
    return envelope * stack([carriers[0], carriers[1], carriers.sum(axis=0) / 2])


topology = Topology.loops([n_nodes])
positions = np.arange(n_nodes) * node_spacing
inside = ((positions >= slab_start) & (positions < slab_start + n_turns * pitch)).astype(float)
directors, loop_helix = helix_media(positions - slab_start)
media = (inside * loop_helix[:, 0] + (1 - inside) * surrounding_response).real()  # [n_nodes] Excitation- <- Field+
node_response = media.field()                                                     # (Excitation- <- Field+)[V]
node_inverse = media.inverse().field()                                            # (Field+ <- Excitation-)[V]
edge_response = surrounding_response.real()                                       # [] Excitation- <- Field+

electric_initial = initial_pulse(positions).field()                               # [3] Field+[V]
shifted_pulse = initial_pulse(positions + node_spacing / 2 + interval / (2 * surrounding_index))
magnetic_initial = (surrounding_index * (real.t.inverse() | (real.z ^ shifted_pulse))).field()  # [3] Field+[E]
stride = n_steps // n_frames
state_stream = islice(
    leapfrog_states(topology.d[0], node_inverse, edge_response, electric_initial, magnetic_initial, node_spacing, interval, n_steps),
    stride - 1,
    None,
    stride,
)
slab_indices = np.flatnonzero(inside)
nodes = positions * real.z.dual()                                                 # [n_nodes] Direction-
optic_axes = directors[slab_indices, 0].real()                                    # [...] Direction-
states = show_states(
    state_stream,
    render.pulse_animation,
    nodes,
    nodes[slab_indices],
    optic_axes,
    2,
    name="coating_pulse",
    duration_ms=frame_duration_ms,
)

Grey rods mark the rotating crystal optic axis inside the translucent slab, while spoke length and saturation follow electric-field amplitude and hue encodes the transverse field angle around the beam axis. Entering from the left as a flat planar wavepacket, the linearly polarized pulse splits at the slab into a reflected co-rotating helix and a transmitted counter-rotating helix.

In [ ]:
initial_energy = electromagnetic_energy(electric_initial, magnetic_initial, node_response, edge_response, node_spacing).sites.sum()  # [3] Scalar+
final_energy = electromagnetic_energy(*states[-1], node_response, edge_response, node_spacing)  # [3] Scalar+[V]
front = topology.reading(np.flatnonzero(positions < slab_start), 0)
back = topology.reading(np.flatnonzero(positions >= slab_start + n_turns * pitch), 0)
pulse_reflectance = (front * final_energy).sites.sum() / initial_energy           # [3] Scalar+
pulse_transmittance = (back * final_energy).sites.sum() / initial_energy          # [3] Scalar+

with np.printoptions(precision=3, suppress=True):
    print("reflected:  ", pulse_reflectance.to_array())
    print("transmitted:", pulse_transmittance.to_array())

## Axion interface reflection

An axion coupling is an intrinsic scalar whose product with a field bivector adds a handed contribution directly to the excitation antibivector. When constant inside a medium, that term cancels from the bulk wave equation and leaves propagation speeds unchanged, whereas a jump across an interface alters the tangential excitation trace and couples orthogonal polarizations even when the dielectric permittivity matches on both sides ([Obukhov and Hehl (2005)](https://arxiv.org/abs/physics/0504172)).

In [ ]:
n_axion_jumps = 241
axion_jumps = np.linspace(-3, 3, n_axion_jumps)

In [ ]:
axion_response = surrounding_response + mv.antipseudoscalar(axion_jumps[:, None]) * Field
axion_outgoing = (field_boundary + excitation_boundary(axion_response))(surrounding_wave)
axion_scatter, _ = interface_scattering(axion_outgoing, surrounding_response, surrounding_index, parallel)
reflected_axion = axion_scatter(mv.xt)                                            # [n_axion_jumps] Polarization+
parallel_reflection = mv.xt * (reflected_axion & (mv.x.dual() & time))            # [n_axion_jumps] Polarization+
perpendicular_reflection = mv.yt * (reflected_axion & (mv.y.dual() & time))       # [n_axion_jumps] Polarization+
axion_reflection = stack([
    wave_power(surrounding_wave(parallel_reflection), surrounding_response),
    wave_power(surrounding_wave(perpendicular_reflection), surrounding_response),
    wave_power(surrounding_wave(reflected_axion), surrounding_response),
]) / wave_power(surrounding_wave(mv.xt), surrounding_response)                    # [3, n_axion_jumps] Scalar+
save_figure(render.axion_figure(axion_jumps, axion_reflection), "coating_axion");

Cross-polarized reflection in orange grows quadratically around zero jump and dominates small discontinuities, co-polarized reflection in blue enters at fourth order, and total reflectance in teal sums both channels.

In [ ]:
# checks
assert ((matching_reflection + matching_transmission - 1).abs() < 1e-10).all()
assert (matching_reflection[1, wavelengths == design_wavelength].abs() < 1e-10).all()
assert ((twisted_reflection + twisted_transmission - 1).abs() < 1e-10).all()
assert ((twisted_reflection[0, 0] - twisted_reflection[1, 1]).abs() < 1e-10).all()
assert ((angle_reflection + angle_transmission - 1).abs() < 1e-10).all()
co_start = interior[0, 0, 0].real().scalar_norm_squared() + (-1j * interior[0, 0, 0]).real().scalar_norm_squared()
co_end = interior[0, -1, -1].real().scalar_norm_squared() + (-1j * interior[0, -1, -1]).real().scalar_norm_squared()
assert (co_end < 0.01 * co_start).all()
assert ((pulse_reflectance + pulse_transmittance - 1).abs() < 1e-2).all()
assert (pulse_reflectance[0] > 0.95).all() and (pulse_reflectance[1] < 0.05).all()
assert ((axion_reflection[0] + axion_reflection[1] - axion_reflection[2]).abs() < 1e-10).all()
assert (axion_reflection[:, axion_jumps == 0].abs() < 1e-12).all()